In [ ]:
import pandas as pd

from i7_hot_reload.hot_reload import hot_reload_a_module_from_cwd

b000_env = hot_reload_module_from_cwd('b000_env.py')

from b000_env import dyr, file, var, const

In [ ]:
# Get all CSV files in the current directory
csv_files = list(dyr.pitchbook.glob('*.csv'))
csv_files

In [ ]:
for csv_file in csv_files:
    # Read the CSV file into a DataFrame
    df = pd.read_csv(csv_file, low_memory = False)

    # Create the corresponding Parquet file path (same name, .parquet extension)
    parquet_file = csv_file.with_suffix('.parquet')

    # Write the DataFrame to Parquet
    df.to_parquet(parquet_file, index = False)

    print(f"Converted {csv_file.name} -> {parquet_file.name}")


In [ ]:
# read all the created parquet files and convert them to excels for easier viewing
parquet_files = list(dyr.pitchbook.glob('*.parquet'))
parquet_files

In [3]:
len(parquet_files)

36

In [ ]:
max_rows = 1_000_000  # Excel row limit (slightly below 1,048,576)

for parquet_file in parquet_files:
    # Read the Parquet file
    df = pd.read_parquet(parquet_file)

    # Convert all columns to string to avoid Excel URL limit warning
    df = df.astype(str)

    base_name = parquet_file.stem

    if len(df) <= max_rows:
        excel_file = parquet_file.with_suffix('.xlsx')
        df.to_excel(excel_file, index = False)
        print(f"Converted {parquet_file.name} -> {excel_file.name}")
    else:
        num_parts = (len(df) // max_rows) + (1 if len(df) % max_rows else 0)
        for i in range(num_parts):
            start = i * max_rows
            end = start + max_rows
            df_part = df.iloc[start:end]

            # Ensure each chunk also has strings
            df_part = df_part.astype(str)

            excel_file = parquet_file.with_name(f"{base_name}_part{i + 1}.xlsx")
            df_part.to_excel(excel_file, index = False)
            print(f"Converted {parquet_file.name} -> {excel_file.name} (rows {start}-{min(end, len(df))})")

Converted FundTeamRelation.parquet -> FundTeamRelation.xlsx
Converted PersonAdvisoryRelation.parquet -> PersonAdvisoryRelation.xlsx
Converted PersonAffiliatedFundRelation.parquet -> PersonAffiliatedFundRelation.xlsx
Converted PersonEducationRelation.parquet -> PersonEducationRelation_part1.xlsx (rows 0-1000000)
Converted PersonEducationRelation.parquet -> PersonEducationRelation_part2.xlsx (rows 1000000-2000000)
Converted PersonEducationRelation.parquet -> PersonEducationRelation_part3.xlsx (rows 2000000-2306319)
Converted CompanyBuySideRelation.parquet -> CompanyBuySideRelation.xlsx
Converted EntityLocationRelation.parquet -> EntityLocationRelation_part1.xlsx (rows 0-1000000)
Converted EntityLocationRelation.parquet -> EntityLocationRelation_part2.xlsx (rows 1000000-1442432)
